# AHR PAS-B Screening — Approved-Drug Repurposing (hierarchical)

Protocol validated by redocking (best pose RMSD **0.49 Å** vs crystal
indirubin in 7ZUB chain D) — see `research/AHR_redocking_result.md`.

**Hierarchical virtual screening** (fast → accurate):
1. **Stage A** — Vina-only breadth screen of all 589 approved drugs
   (`--cnn_scoring none --exhaustiveness 4`), chunked + parallel.
2. **Stage B** — CNN refinement (`--cnn fast --cnn_scoring rescore`,
   `--exhaustiveness 24`) of the **top 50** Vina hits.
3. **Stage C** — rank by CNN affinity, check pocket contacts vs
   His337 / Gln383 / Tyr322.

Library: 589 approved drugs, MaxMin diversity pick from 2,838 dockable
ChEMBL `max_phase=4` molecules, protonated pH 7.4, ETKDGv3 3D.

**Run on GPU:** Runtime → T4 GPU.

In [ ]:
# 1. GNINA + dependencies
!wget -q https://github.com/gnina/gnina/releases/download/v1.3.3/gnina.cuda12.8.static -O gnina
!chmod +x gnina
!apt-get -qq update && apt-get -qq install -y openbabel 2>/dev/null | tail -1
!pip install -q rdkit biopython pandas
!./gnina --version

In [ ]:
# 2. Receptor prep - identical to the validated redocking
!wget -q https://files.rcsb.org/download/7ZUB.pdb

from Bio.PDB import PDBParser, PDBIO, Select
s = PDBParser(QUIET=True).get_structure('7ZUB', '7ZUB.pdb')
class ProtOnly(Select):
    def accept_chain(self, c): return c.id == 'D'
    def accept_residue(self, r): return r.id[0] == ' '
class LigOnly(Select):
    def accept_residue(self, r): return r.get_resname() == 'JY6'
io = PDBIO(); io.set_structure(s)
io.save('receptor.pdb', ProtOnly())
io.save('indirubin.pdb', LigOnly())
!obabel receptor.pdb -xr -h -p 7.4 -O receptor_prep.pdb 2>/dev/null
!echo "receptor_prep: $(grep -c '^ATOM' receptor_prep.pdb) atoms"

In [ ]:
# 3. Fetch the screening library (secret gist; ChEMBL approved drugs)
!wget -q https://gist.githubusercontent.com/skadlem/3b89119ebd49a7cfa0dd5ad6d64ea903/raw/library_batch1.sdf -O library.sdf
from rdkit import Chem
n = sum(1 for _ in Chem.SDMolSupplier('library.sdf', removeHs=True, sanitize=False))
print(f'library.sdf: {n} compounds')

In [ ]:
# 4. STAGE A - Vina-only breadth screen (no CNN), chunked + parallel
# Rationale: MC sampling is CPU-bound on 2 vCPUs; dropping CNN rescoring and
# running 2 single-threaded gnina processes in parallel gives a big speedup.
# Chunks = checkpoints: a lost session only loses the current chunk.
import subprocess, time, os
from concurrent.futures import ThreadPoolExecutor

raw = open('library.sdf').read()
recs = [r for r in raw.split('$$$$\n') if r.strip()]
CH, NPROC = 100, 2
nchunks = (len(recs) + CH - 1) // CH
for k in range(nchunks):
    with open(f'lib_{k:02d}.sdf', 'w') as f:
        f.write('$$$$\n'.join(recs[k*CH:(k+1)*CH]) + '$$$$\n')
print(f'{len(recs)} compounds -> {nchunks} chunks of ~{CH}', flush=True)

def run(k):
    out = f'stageA_{k:02d}.sdf.gz'
    env = {**os.environ, 'OMP_NUM_THREADS': '1'}
    t = time.time()
    r = subprocess.run(['./gnina', '-r', 'receptor_prep.pdb', '-l', f'lib_{k:02d}.sdf',
                        '--autobox_ligand', 'indirubin.pdb', '--autobox_add', '8',
                        '--cnn_scoring', 'none',
                        '--exhaustiveness', '4', '--num_modes', '1',
                        '--cpu', '1', '-o', out],
                       capture_output=True, text=True, timeout=5400, env=env)
    return k, r.returncode, time.time() - t, out, (r.stderr or '')[-300:]

t0 = time.time()
bad = 0
with ThreadPoolExecutor(max_workers=NPROC) as ex:
    for k, rc, dt, out, err in ex.map(run, range(nchunks)):
        if rc != 0:
            bad += 1
            print(f'chunk {k:02d}: FAILED rc={rc}\n{err}', flush=True)
        else:
            print(f'chunk {k:02d}: ok {dt/60:.1f}min ({out}) elapsed={(time.time()-t0)/60:.1f}min', flush=True)
print(f'STAGE A done: {nchunks - bad}/{nchunks} chunks ok in {(time.time()-t0)/60:.1f} min', flush=True)

In [ ]:
# 5. STAGE B - CNN refinement of the top Vina hits
# Best Vina affinity per compound from stage A (most negative = best), then
# re-dock the shortlist with CNN rescoring at higher exhaustiveness.
from rdkit import Chem
import glob, subprocess, time, pandas as pd

best = {}
for f in sorted(glob.glob('stageA_*.sdf.gz')):
    for m in Chem.SDMolSupplier(f, removeHs=True, sanitize=False):
        if m is None:
            continue
        cid = m.GetProp('_Name')
        if not m.HasProp('Affinity'):
            continue
        aff = float(m.GetProp('Affinity'))
        if cid not in best or aff < best[cid][0]:
            best[cid] = (aff, m)

print(f'{len(best)} unique compounds scored in stage A')
rows = [(cid, aff, m.GetProp('name') if m.HasProp('name') else cid)
        for cid, (aff, m) in best.items()]
pd.DataFrame(rows, columns=['id', 'vina_affinity', 'name']).sort_values(
    'vina_affinity').to_csv('stageA_results.csv', index=False)
print('wrote stageA_results.csv (all compounds, Vina affinity)')

TOPN = 50
ranked = sorted(best.items(), key=lambda kv: kv[1][0])[:TOPN]
w = Chem.SDWriter('shortlist.sdf')
for cid, (aff, m) in ranked:
    w.write(m)
w.close()
print(f'shortlist: {len(ranked)} top Vina hits -> shortlist.sdf', flush=True)

t = time.time()
r = subprocess.run(['./gnina', '-r', 'receptor_prep.pdb', '-l', 'shortlist.sdf',
                    '--autobox_ligand', 'indirubin.pdb', '--autobox_add', '8',
                    '--cnn', 'fast', '--cnn_scoring', 'rescore',
                    '--exhaustiveness', '24', '--num_modes', '3',
                    '-o', 'refined.sdf.gz'],
                   capture_output=True, text=True, timeout=5400)
print(f'STAGE B rc={r.returncode} in {(time.time()-t)/60:.1f} min')
if r.returncode != 0:
    print((r.stderr or '')[-500:])

In [ ]:
# 6. Rank refined hits + pocket contacts (His337 closest; Gln383/Tyr322 H-bonds)
from rdkit import Chem
from Bio.PDB import PDBParser
import numpy as np, pandas as pd

KEY = {'HIS337', 'GLN383', 'TYR322', 'LEU308', 'ILE325', 'LEU353', 'PHE351', 'PHE287'}

rec = PDBParser(QUIET=True).get_structure('R', 'receptor_prep.pdb')[0]
res_atoms = [(res.get_resname() + str(res.id[1]),
              np.array([a.coord for a in res.get_atoms()])) for res in rec.get_residues()]

def g(m, k):
    return float(m.GetProp(k)) if m.HasProp(k) else None

best_per = {}
for m in Chem.SDMolSupplier('refined.sdf.gz', removeHs=True, sanitize=False):
    if m is None:
        continue
    cid = m.GetProp('_Name')
    px = m.GetConformer().GetPositions()
    contacts = []
    for name, xyz in res_atoms:
        d = float(np.linalg.norm(xyz - px, axis=1).min())
        if d <= 4.5:
            contacts.append((name, round(d, 1)))
    contacts.sort(key=lambda t: t[1])
    cnn_aff = g(m, 'CNNaffinity')
    info = {
        'id': cid,
        'name': m.GetProp('name') if m.HasProp('name') else cid,
        'cnn_aff': cnn_aff,
        'cnn_pose': g(m, 'CNNscore'),
        'vina': g(m, 'Affinity'),
        'n_contacts': len(contacts),
        'contacts': contacts,
    }
    key = info['cnn_aff'] if info['cnn_aff'] is not None else info['vina']
    if cid not in best_per or key > best_per[cid]['_rankkey']:
        info['_rankkey'] = key
        best_per[cid] = info

ranked = sorted(best_per.values(), key=lambda r: -r['_rankkey'])
print(f'{len(ranked)} refined compounds\n')
hdr = f"{'CHEMBL':<10}{'name':<22}{'CNNaff':>7}{'CNNpose':>8}{'vina':>7}{'#cont':>6}  key-contacts"
print(hdr); print('-' * len(hdr))
for r in ranked[:20]:
    kc = [n for n, d in r['contacts'] if n in KEY]
    ca = f"{r['cnn_aff']:>7.2f}" if r['cnn_aff'] is not None else '     --'
    cp = f"{r['cnn_pose']:>8.3f}" if r['cnn_pose'] is not None else '       --'
    print(f"{r['id']:<10}{r['name'][:21]:<22}{ca}{cp}"
          f"{r['vina']:>7.2f}{r['n_contacts']:>6}  {' '.join(kc[:6])}")

df = pd.DataFrame([{k: v for k, v in r.items() if k not in ('contacts', '_rankkey')}
                   for r in ranked])
df['top_contacts'] = [' '.join(f'{n}({d})' for n, d in r['contacts'][:8]) for r in ranked]
df.to_csv('screen_results.csv', index=False)
print('\nwrote screen_results.csv')